In [1]:
import pandas as pd
import numpy as np

# ==========================================
# 1. Load Test Predictions
# ==========================================

pred = pd.read_csv(
    "data/final_test_predictions.csv",
    parse_dates=["Date"],
    index_col="Date"
)

market = pd.read_csv(
    "data/market_data_final_features.csv",
    parse_dates=["Date"],
    index_col="Date"
)

df = pred.join(
    market[
        [
            "SP500",
            "VIX",
            "Future_Drawdown_10D"
        ]
    ],
    how="left"
)

print(
    "Backtest Period:",
    df.index.min(),
    "to",
    df.index.max()
)

print(
    "Total Test Days:",
    len(df)
)

# ==========================================
# 2. Risk Levels
# ==========================================

WARNING_THRESHOLD = 0.37
HIGH_THRESHOLD = 0.62

df["Risk_Level"] = np.select(
    [
        df["Crash_Probability"] >= HIGH_THRESHOLD,
        df["Crash_Probability"] >= WARNING_THRESHOLD
    ],
    [
        "HIGH",
        "WARNING"
    ],
    default="LOW"
)

print("\nRisk Level Distribution:")
print(
    df["Risk_Level"].value_counts()
)

# ==========================================
# 3. Detect Crash Episodes
# ==========================================

# Consecutive Crash_Risk=1 dates are treated
# as one crash-risk episode.

crash = df["Actual"].eq(1)

episode_start = (
    crash &
    ~crash.shift(1, fill_value=False)
)

episode_id = episode_start.cumsum()

df["Crash_Episode"] = np.where(
    crash,
    episode_id,
    np.nan
)

episodes = []

for ep in df["Crash_Episode"].dropna().unique():

    ep_data = df[
        df["Crash_Episode"] == ep
    ]

    start_date = ep_data.index.min()
    end_date = ep_data.index.max()

    # Look at 30 trading observations ending
    # on the first crash-risk date.
    position = df.index.get_loc(start_date)

    lookback_start = max(
        0,
        position - 30
    )

    warning_window = df.iloc[
        lookback_start:position + 1
    ]

    warning_signals = warning_window[
        warning_window["Crash_Probability"]
        >= WARNING_THRESHOLD
    ]

    high_signals = warning_window[
        warning_window["Crash_Probability"]
        >= HIGH_THRESHOLD
    ]

    first_warning = (
        warning_signals.index.min()
        if not warning_signals.empty
        else pd.NaT
    )

    first_high = (
        high_signals.index.min()
        if not high_signals.empty
        else pd.NaT
    )

    # Trading-day lead time
    warning_lead = (
        position -
        df.index.get_loc(first_warning)
        if pd.notna(first_warning)
        else np.nan
    )

    high_lead = (
        position -
        df.index.get_loc(first_high)
        if pd.notna(first_high)
        else np.nan
    )

    episodes.append({

        "Episode_Start":
            start_date,

        "Episode_End":
            end_date,

        "Crash_Risk_Days":
            len(ep_data),

        "Worst_10D_Drawdown":
            ep_data[
                "Future_Drawdown_10D"
            ].min(),

        "Max_Probability_Before_Start":
            warning_window[
                "Crash_Probability"
            ].max(),

        "First_Warning_Date":
            first_warning,

        "Warning_Lead_Trading_Days":
            warning_lead,

        "First_High_Risk_Date":
            first_high,

        "High_Risk_Lead_Trading_Days":
            high_lead
    })

episode_df = pd.DataFrame(
    episodes
)

print(
    "\n========== CRASH EPISODES =========="
)

if not episode_df.empty:

    print(
        episode_df.to_string(
            index=False
        )
    )

else:

    print(
        "No crash episodes in test period."
    )

# ==========================================
# 4. Overall Warning Performance
# ==========================================

warning_pred = (
    df["Crash_Probability"]
    >= WARNING_THRESHOLD
)

high_pred = (
    df["Crash_Probability"]
    >= HIGH_THRESHOLD
)

actual = df["Actual"].eq(1)

print(
    "\n========== WARNING THRESHOLD 0.37 =========="
)

print(
    "Actual Crash-Risk Days:",
    actual.sum()
)

print(
    "Detected Crash-Risk Days:",
    (warning_pred & actual).sum()
)

print(
    "Missed Crash-Risk Days:",
    (~warning_pred & actual).sum()
)

print(
    "False Alert Days:",
    (warning_pred & ~actual).sum()
)

print(
    "\n========== HIGH THRESHOLD 0.62 =========="
)

print(
    "Detected Crash-Risk Days:",
    (high_pred & actual).sum()
)

print(
    "Missed Crash-Risk Days:",
    (~high_pred & actual).sum()
)

print(
    "False Alert Days:",
    (high_pred & ~actual).sum()
)

# ==========================================
# 5. Highest Risk Predictions
# ==========================================

print(
    "\n========== TOP 20 MODEL WARNINGS =========="
)

print(

    df[
        [
            "SP500",
            "VIX",
            "Crash_Probability",
            "Risk_Level",
            "Actual",
            "Future_Drawdown_10D"
        ]
    ]

    .sort_values(
        "Crash_Probability",
        ascending=False
    )

    .head(20)

    .to_string()

)

# ==========================================
# 6. Save Results
# ==========================================

df.to_csv(
    "data/historical_backtest_daily.csv"
)

episode_df.to_csv(
    "data/crash_episode_backtest.csv",
    index=False
)

print(
    "\nSaved:"
    "\ndata/historical_backtest_daily.csv"
    "\ndata/crash_episode_backtest.csv"
)

Backtest Period: 2021-10-15 00:00:00 to 2026-07-02 00:00:00
Total Test Days: 1185

Risk Level Distribution:
Risk_Level
LOW        945
WARNING    148
HIGH        92
Name: count, dtype: int64

========== CRASH EPISODES ==========
Episode_Start Episode_End  Crash_Risk_Days  Worst_10D_Drawdown  Max_Probability_Before_Start First_Warning_Date  Warning_Lead_Trading_Days First_High_Risk_Date  High_Risk_Lead_Trading_Days
   2022-01-11  2022-01-14                4           -0.084598                      0.579067         2021-11-30                       29.0                  NaT                          NaN
   2022-02-09  2022-02-09                1           -0.078846                      0.720841         2022-01-11                       20.0           2022-01-31                          7.0
   2022-04-19  2022-04-20                2           -0.074017                      0.798514         2022-03-09                       28.0           2022-03-09                         28.0
   2022-04-25  2